# 03 - AI re-coding of refund reasons

Problem: 42% of refunds are GW-OTHER (first option in the dropdown), 879 above the Rs 500 goodwill cap. Is that real goodwill over the cap, or a real reason (return, failed payment...) with a lazy code?

Plan:
1. pull a stratified sample -> **I label it by hand first** (gold set)
2. keyword baseline (no AI) -> what the AI has to beat
3. LLM with prompt v1, cached, scored on the gold set

In [1]:
import os, json, re
import pandas as pd

pd.set_option('display.max_colwidth', 120)

df = pd.read_csv('output/tickets_clean.csv')
refunds = df[df.refund_inr.notna()].copy()

CODES = ['GW-OTHER', 'DOA-REPL', 'LOST-TRANSIT', 'DUP-PAYMENT', 'CANCEL',
         'PRICE-ADJ', 'RETURN-QC-OK', 'WTY-BUYBACK']

print('refunds:', len(refunds), '| GW-OTHER:', (refunds.refund_reason_code == 'GW-OTHER').sum())

refunds: 2340 | GW-OTHER: 991


## 1. Gold set sample (60 tickets)

- 25 GW-OTHER above cap (the money question)
- 10 GW-OTHER at/below cap (probably real goodwill - checks the model doesn't re-code everything)
- 15 other codes (agent picked something specific - model should mostly agree)
- 10 refund + replacement

File is only written once so my labels never get overwritten.

In [2]:
GOLD_PATH = 'eval/gold_set.csv'

gw = refunds.refund_reason_code == 'GW-OTHER'
over = refunds.flag_gw_over_cap
both = refunds.flag_refund_and_repl

strata = [
    ('gw_over_cap', refunds[gw & over & ~both], 25),
    ('gw_under_cap', refunds[gw & ~over & ~both], 10),
    ('other_code', refunds[~gw & ~both], 15),
    ('refund_and_repl', refunds[both], 10),
]

if not os.path.exists(GOLD_PATH):
    gold = pd.concat([d.sample(n, random_state=42).assign(stratum=name) for name, d, n in strata])
    gold = gold[['ticket_id', 'stratum', 'category', 'refund_inr', 'refund_reason_code',
                 'replacement_issued', 'customer_message', 'agent_notes']]
    gold['my_code'] = ''          # fill by hand: one of CODES or UNCLEAR
    gold['my_real_goodwill'] = '' # Y if it's a genuine goodwill gesture, N otherwise
    gold.to_csv(GOLD_PATH, index=False)
    print('written', GOLD_PATH)

gold = pd.read_csv(GOLD_PATH, keep_default_na=False)
print(gold.stratum.value_counts())
print('labelled so far:', (gold.my_code != '').sum(), '/', len(gold))

written eval/gold_set.csv
stratum
gw_over_cap        25
other_code         15
gw_under_cap       10
refund_and_repl    10
Name: count, dtype: int64
labelled so far: 0 / 60


## 2. Keyword baseline

Simple rules on the agent note (+ message). First match wins, order matters.

In [3]:
RULES = [
    ('CANCEL',       r'cancel'),
    ('DUP-PAYMENT',  r'deduct|twice|double (charge|payment)|failed ord|without ord|utr|payment gateway'),
    ('LOST-TRANSIT', r'lost|not received|undeliver|never arrived'),
    ('DOA-REPL',     r'\bdoa\b|dead on arrival|dead|not (turning|switching) on'),
    ('WTY-BUYBACK',  r'buy.?back'),
    ('PRICE-ADJ',    r'price|coupon|discount'),
    ('RETURN-QC-OK', r'\bqc\b|reverse pick|return'),
    ('GW-OTHER',     r'goodwill|gesture|compensat'),
]

def keyword_code(note, msg):
    text = f'{note} {msg}'.lower()
    for code, pat in RULES:
        if re.search(pat, text):
            return code
    return 'UNCLEAR'

refunds['kw_code'] = [keyword_code(n, m) for n, m in zip(refunds.agent_notes, refunds.customer_message)]

In [4]:
# rough check before hand labels: on non-GW rows the agent chose a specific code, so use it as a proxy truth
specific = refunds[refunds.refund_reason_code != 'GW-OTHER']
print('keyword agrees with agent code on specific codes:', (specific.kw_code == specific.refund_reason_code).mean().round(3))
pd.crosstab(specific.refund_reason_code, specific.kw_code)

keyword agrees with agent code on specific codes: 0.523


kw_code,CANCEL,DOA-REPL,DUP-PAYMENT,GW-OTHER,LOST-TRANSIT,PRICE-ADJ,RETURN-QC-OK,UNCLEAR
refund_reason_code,,,,,,,,
CANCEL,216,0,0,0,0,0,0,6
DOA-REPL,0,2,8,7,7,2,19,102
DUP-PAYMENT,0,0,257,0,0,1,0,63
LOST-TRANSIT,0,0,0,0,10,2,1,34
PRICE-ADJ,0,0,0,0,0,64,0,7
RETURN-QC-OK,0,0,107,4,35,2,156,148
WTY-BUYBACK,0,2,4,7,5,1,12,58


In [5]:
# what the keyword baseline thinks the GW-OTHER refunds really are
g = refunds[refunds.refund_reason_code == 'GW-OTHER']
g.groupby('kw_code').refund_inr.agg(['count', 'sum']).sort_values('count', ascending=False)

,count,sum
kw_code,,
UNCLEAR,372,1130989.0
DUP-PAYMENT,236,638718.0
RETURN-QC-OK,129,381013.0
CANCEL,120,355854.0
LOST-TRANSIT,51,171666.0
PRICE-ADJ,41,81260.0
DOA-REPL,22,63192.0
GW-OTHER,20,84344.0


In [6]:
# where it breaks - read some
g[['ticket_id', 'kw_code', 'agent_notes']].sample(8, random_state=7)

,ticket_id,kw_code,agent_notes
2294,TK-242877,DUP-PAYMENT,1. failed order after payment\n2. conf with payment gateway\n3. refund of rs 1169 initiated to source
11164,TK-253989,UNCLEAR,"cx: rapid batery drain | advised disabling anc/aod to test -> advised settings change, cx satisfied.\nReversed the p..."
1877,TK-242358,UNCLEAR,resolved on call ~Manish
5667,TK-247128,UNCLEAR,customer reached out - unit rcvd damaged. chk w/ crr for damage claim. full rfnd processed (1105).
300,TK-240381,CANCEL,"cx reached out - cancellation request. Cancelled before dispatch. Already shipped, advised RTO/return. Cx satisfied ..."
10745,TK-253466,UNCLEAR,"rfnd approved, rs 712, 5-7 working days"
7747,TK-249730,UNCLEAR,"reset done, paired ok. issue: device not discoverable. asked cx to reset the device. issued refund + replacement bot..."
278,TK-240354,CANCEL,cx: cancel ord | chk shipment satus -> refunded rs 4248


## 3. LLM labelling (prompt v1)

Gemini 2.5 Flash-Lite (cheap, free tier). temperature 0, JSON output. Every answer is cached by ticket_id + prompt version, so re-running costs nothing and the final tool runs without a key.

In [7]:
MODEL = 'gemini-2.5-flash-lite'
PROMPT_VERSION = 'v1'
CACHE_PATH = 'cache/llm_labels.jsonl'

PROMPT = open(f'prompts/{PROMPT_VERSION}.txt', encoding='utf-8').read()

def load_cache():
    cache = {}
    if os.path.exists(CACHE_PATH):
        with open(CACHE_PATH, encoding='utf-8') as f:
            for line in f:
                rec = json.loads(line)
                cache[(rec['ticket_id'], rec['prompt_version'])] = rec
    return cache

cache = load_cache()
len(cache)

0

In [8]:
def call_llm(client, row):
    from google.genai import types
    prompt = PROMPT.format(customer_message=row.customer_message, agent_notes=row.agent_notes)
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0, response_mime_type='application/json'),
    )
    out = json.loads(resp.text)
    return {
        'ticket_id': row.ticket_id,
        'prompt_version': PROMPT_VERSION,
        'model': MODEL,
        **out,
        'in_tokens': resp.usage_metadata.prompt_token_count,
        'out_tokens': resp.usage_metadata.candidates_token_count,
    }

def label(rows):
    todo = [r for r in rows.itertuples() if (r.ticket_id, PROMPT_VERSION) not in cache]
    print(f'{len(rows) - len(todo)} cached, {len(todo)} to call')
    if not todo:
        return
    if not os.environ.get('GEMINI_API_KEY'):
        print('no GEMINI_API_KEY -> skipping, using cache only')
        return
    from google import genai
    client = genai.Client()
    with open(CACHE_PATH, 'a', encoding='utf-8') as f:
        for r in todo:
            try:
                rec = call_llm(client, r)
            except Exception as e:
                print(r.ticket_id, 'failed:', str(e)[:120])
                if 'API_KEY_INVALID' in str(e):
                    break
                continue
            cache[(r.ticket_id, PROMPT_VERSION)] = rec
            f.write(json.dumps(rec) + '\n')

In [9]:
# gold set first - no point running 991 rows before we know the accuracy
label(refunds[refunds.ticket_id.isin(gold.ticket_id)])

0 cached, 60 to call


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


TK-240338 failed: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'I


## 4. Score on gold set

Runs once `my_code` is filled in eval/gold_set.csv.

In [ ]:
# labels are done in eval/gold_set_labelling.xlsx (dropdowns), copied back to the csv here
XLSX = 'eval/gold_set_labelling.xlsx'
if os.path.exists(XLSX):
    labels = pd.read_excel(XLSX, sheet_name='label_me', dtype=str).fillna('')
    gold = (pd.read_csv(GOLD_PATH, keep_default_na=False)
              .drop(columns=['my_code', 'my_real_goodwill', 'my_comment'], errors='ignore'))
    gold = gold.merge(labels[['ticket_id', 'my_code', 'my_real_goodwill', 'my_comment']], on='ticket_id', how='left')
    gold.to_csv(GOLD_PATH, index=False)

gold = pd.read_csv(GOLD_PATH, keep_default_na=False)
done = gold[gold.my_code != ''].copy()

if len(done) == 0:
    print('gold set not labelled yet')
else:
    done['kw_code'] = [keyword_code(n, m) for n, m in zip(done.agent_notes, done.customer_message)]
    done['llm_code'] = [cache.get((t, PROMPT_VERSION), {}).get('code', '') for t in done.ticket_id]

    scores = pd.DataFrame({
        'agent (original)': (done.refund_reason_code == done.my_code),
        'keyword': (done.kw_code == done.my_code),
        f'llm {PROMPT_VERSION}': (done.llm_code == done.my_code),
    })
    print('n =', len(done))
    print(scores.mean().round(3))
    print(scores.groupby(done.stratum).mean().round(2))

## Notes

(write after labelling + first LLM run)